# Baseline YOLOv8 model: invasive plant classification (Group 9)

Classes (folder names): cow_parsnip, false_solomons_seal, fireweed, giant_hogweed, himalayan_balsam, japanese_knotweed, lewis_monkeyflower, purple_loosestrife

**Baseline hyperparameters (teammates change ONE of these, 3 different values each):**

| Parameter | Baseline value |
|---|---|
| model | yolov8n-cls.pt (ImageNet pretrained) |
| epochs | 100 |
| patience | 15 |
| imgsz | 640 |
| batch | 16 |
| lr0 | 0.01 |
| optimizer | SGD |
| seed | 0 |

Before running: open this notebook in local Jupyter with CUDA-enabled PyTorch.
Dataset expected: `C:/dataset/` containing `train/`, `valid/` (or `val/`), and `test/` species folders.


In [1]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

import torch
assert torch.cuda.is_available(), "CUDA is unavailable. Install CUDA-enabled PyTorch in this notebook environment."
print(torch.cuda.get_device_name(0))


Ultralytics 8.4.204  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
Setup complete  (20 CPUs, 15.8 GB RAM, 822.6/930.4 GB disk)
NVIDIA GeForce RTX 3060 Ti


## 1. Use the local training/val/test data

In [2]:
import os, subprocess

LOCAL_DIR = "C:/dataset"

# Windows directory junction: val points to valid without copying or moving images.
if not os.path.isdir(f"{LOCAL_DIR}/val") and os.path.isdir(f"{LOCAL_DIR}/valid"):
    subprocess.run(["cmd", "/c", "mklink", "/J",
                    os.path.normpath(f"{LOCAL_DIR}/val"),
                    os.path.normpath(f"{LOCAL_DIR}/valid")], check=True)

print(os.listdir(LOCAL_DIR))


['README.roboflow.txt', 'test', 'train', 'train.cache', 'val', 'val.cache', 'valid', '_metadata']


## 2. Check the local classification folders

In [3]:
# Classification uses species folders instead of data.yaml and box labels.
print(os.listdir("C:/dataset/train"))


['cow_parsnip', 'false_solomons_seal', 'fireweed', 'giant_hogweed', 'himalayan_balsam', 'japanese_knotweed', 'lewis_monkeyflower', 'purple_loosestrife']


## 3. Sanity checks (do not skip: catches wrong classes and missing files)

In [4]:
import glob, os
names = sorted(os.path.basename(p) for p in glob.glob("C:/dataset/train/*") if os.path.isdir(p))
print(names)


['cow_parsnip', 'false_solomons_seal', 'fireweed', 'giant_hogweed', 'himalayan_balsam', 'japanese_knotweed', 'lewis_monkeyflower', 'purple_loosestrife']


In [5]:
import glob, os
from collections import Counter

NAMES = names

for split in ["train", "val", "test"]:
    imgs = glob.glob(f"C:/dataset/{split}/*/*.*")
    imgs_with = Counter(os.path.basename(os.path.dirname(p)) for p in imgs)
    print(f"--- {split}: {len(imgs)} images")
    print("   images per class:", {c: imgs_with[c] for c in NAMES})


--- train: 5999 images
   images per class: {'cow_parsnip': 756, 'false_solomons_seal': 778, 'fireweed': 672, 'giant_hogweed': 735, 'himalayan_balsam': 795, 'japanese_knotweed': 711, 'lewis_monkeyflower': 796, 'purple_loosestrife': 756}
--- val: 1099 images
   images per class: {'cow_parsnip': 140, 'false_solomons_seal': 139, 'fireweed': 132, 'giant_hogweed': 143, 'himalayan_balsam': 141, 'japanese_knotweed': 133, 'lewis_monkeyflower': 143, 'purple_loosestrife': 128}
--- test: 239 images
   images per class: {'cow_parsnip': 34, 'false_solomons_seal': 40, 'fireweed': 15, 'giant_hogweed': 35, 'himalayan_balsam': 25, 'japanese_knotweed': 26, 'lewis_monkeyflower': 40, 'purple_loosestrife': 24}


Expected: train 5,999 images, val 1,099, test 239.
Check that all three splits contain all eight classes. A class with no test images gets no meaningful test score.

## 4. Train the baseline

In [6]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")   # pretrained ImageNet weights = transfer learning

results = model.train(
    data="C:/dataset",
    device=0,
    workers=0,   # avoid Windows notebook multiprocessing issues
    epochs=100,
    patience=15,
    imgsz=640,
    batch=16,
    lr0=0.01,
    optimizer="SGD",
    seed=0,
    project="C:/classification_local/runs",   # saved locally
    name="baseline",
    exist_ok=True,
)

Ultralytics 8.4.204  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline, nbs=64, nms=None, opset=None, optimize=False, opt

## 5. Evaluate on OUR 239 test photos (the baseline score you report)

In [7]:
best = YOLO("C:/classification_local/runs/baseline/weights/best.pt")
metrics = best.val(
    data="C:/dataset",
    split="test",
    device=0,
    workers=0,
    project="C:/classification_local/runs",
    name="baseline_test",
    exist_ok=True,
    plots=True,
)

print("Top-1 accuracy:", round(metrics.top1, 4))
print("Top-5 accuracy:", round(metrics.top5, 4))


Ultralytics 8.4.204  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,445,128 parameters, 0 gradients, 3.3 GFLOPs
train: C:\dataset\train... found 5999 images in 8/8 classes  
val: C:\dataset\val... found 1099 images in 8/8 classes  
test: C:\dataset\test... found 239 images in 8/8 classes  
test: Fast image access  (ping: 0.1±0.1 ms, read: 124.0±58.9 MB/s, size: 572.3 KB)
test: Scanning C:\dataset\test... 239 images, 0 corrupt: 100% ━━━━━━━━━━━━ 239/239 1.4Kit/s 0.2s
test: New cache created: C:\dataset\test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 15/15 1.6it/s 9.1s
                   all      0.954      0.996
Speed: 0.2ms preprocess, 1.2ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to C:\classification_local\runs\baseline_test
Top-1 accuracy: 0.954
Top-5 accuracy: 0.9958


In [8]:
import glob, random, cv2
import matplotlib.pyplot as plt

random.seed(1)
imgs = random.sample(sorted(glob.glob("C:/dataset/test/*/*.*")), 4)

fig, axes = plt.subplots(len(imgs), 2, figsize=(12, 5 * len(imgs)))
for row, p in enumerate(imgs):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    gt = os.path.basename(os.path.dirname(p))
    pred = best.predict(p, verbose=False)[0]
    label = pred.names[pred.probs.top1]
    axes[row, 0].imshow(im); axes[row, 0].set_title(f"Your label: {gt}"); axes[row, 0].axis("off")
    axes[row, 1].imshow(im); axes[row, 1].set_title(f"Prediction: {label}"); axes[row, 1].axis("off")
plt.tight_layout(); plt.show()


Output hidden; open in https://colab.research.google.com to view.

## 6. Save metrics and example predictions for the report

In [9]:
import json

out = {
    "run": "baseline",
    "hyperparameters": {"model":"yolov8n-cls.pt","epochs":100,"patience":15,"imgsz":640,"batch":16,"lr0":0.01,"optimizer":"SGD","seed":0},
    "test_metrics": {
        "top1_accuracy": float(metrics.top1),
        "top5_accuracy": float(metrics.top5),
    },
}
json.dump(out, open("C:/classification_local/runs/baseline_metrics.json", "w"), indent=2)
print(json.dumps(out, indent=2))

# example predictions on test photos (for the 'shortcomings' section)
best.predict(sorted(glob.glob("C:/dataset/test/*/*.*")), save=True,
             project="C:/classification_local/runs", name="baseline_predictions", exist_ok=True)


{
  "run": "baseline",
  "hyperparameters": {
    "model": "yolov8n-cls.pt",
    "epochs": 100,
    "patience": 15,
    "imgsz": 640,
    "batch": 16,
    "lr0": 0.01,
    "optimizer": "SGD",
    "seed": 0
  },
  "test_metrics": {
    "top1_accuracy": 0.9539749026298523,
    "top5_accuracy": 0.9958158731460571
  }
}

0: 640x640 cow_parsnip 1.00, false_solomons_seal 0.00, giant_hogweed 0.00, lewis_monkeyflower 0.00, japanese_knotweed 0.00, 2.9ms
1: 640x640 cow_parsnip 1.00, japanese_knotweed 0.00, false_solomons_seal 0.00, himalayan_balsam 0.00, giant_hogweed 0.00, 2.9ms
2: 640x640 cow_parsnip 1.00, giant_hogweed 0.00, false_solomons_seal 0.00, japanese_knotweed 0.00, himalayan_balsam 0.00, 2.9ms
3: 640x640 cow_parsnip 1.00, giant_hogweed 0.00, japanese_knotweed 0.00, himalayan_balsam 0.00, false_solomons_seal 0.00, 2.9ms
4: 640x640 cow_parsnip 1.00, giant_hogweed 0.00, false_solomons_seal 0.00, japanese_knotweed 0.00, lewis_monkeyflower 0.00, 2.9ms
5: 640x640 japanese_knotweed 0.66, gi

[ultralytics.engine.results.Results object with attributes:
 
 boxes: None
 depth: None
 keypoints: None
 masks: None
 names: {0: 'cow_parsnip', 1: 'false_solomons_seal', 2: 'fireweed', 3: 'giant_hogweed', 4: 'himalayan_balsam', 5: 'japanese_knotweed', 6: 'lewis_monkeyflower', 7: 'purple_loosestrife'}
 obb: None
 orig_img: array([[[ 93, 115, 150],
         [122, 144, 179],
         [126, 145, 182],
         ...,
         [101, 183, 141],
         [103, 181, 140],
         [103, 181, 140]],
 
        [[ 86, 108, 143],
         [109, 131, 166],
         [118, 137, 174],
         ...,
         [ 96, 180, 138],
         [ 97, 179, 137],
         [ 96, 178, 136]],
 
        [[106, 126, 161],
         [110, 130, 165],
         [105, 124, 161],
         ...,
         [ 86, 178, 135],
         [ 86, 176, 133],
         [ 85, 175, 132]],
 
        ...,
 
        [[ 13,  24,  46],
         [ 19,  30,  52],
         [ 25,  34,  61],
         ...,
         [ 64, 177, 127],
         [ 71, 182, 132]

In [10]:
import glob
from collections import Counter

names = best.names
hit, total = Counter(), Counter()

for p in sorted(glob.glob("C:/dataset/test/*/*.*")):
    main = os.path.basename(os.path.dirname(p))
    result = best.predict(p, verbose=False)[0]
    pred = result.names[result.probs.top1]
    total[main] += 1
    hit[main] += main == pred

for c, n in names.items():
    print(f"{n:20s} right class in {hit[n]}/{total[n]} photos")
print("Overall:", sum(hit.values()), "/", sum(total.values()))


cow_parsnip          right class in 28/34 photos
false_solomons_seal  right class in 39/40 photos
fireweed             right class in 15/15 photos
giant_hogweed        right class in 33/35 photos
himalayan_balsam     right class in 24/25 photos
japanese_knotweed    right class in 25/26 photos
lewis_monkeyflower   right class in 40/40 photos
purple_loosestrife   right class in 24/24 photos
Overall: 228 / 239


## 7. After it finishes

Everything is in `C:/classification_local/runs/`:
- `baseline/weights/best.pt`, `baseline/results.png` (training curves)
- `baseline_test/` (confusion matrix on the test set)
- `baseline_metrics.json`, `baseline_predictions/`

Put these in the classification folder of the repo, and send teammates the hyperparameter table above.